## 1. N-gram (KN 3-gram)
**Paper / idea:** Bengio 2003 / Kneser-Ney 1995

Count-based baseline. Kneser-Ney smoothed trigram on BPE tokens. No gradients, no embeddings: shows what pure statistics give you and why sparsity motivates neural LMs.

**Setup:** TinyStories, 4096-token BPE, context 256, ~10M parameters, 4000 steps x 32 x 256 tokens, fp16 on a T4.
Run top to bottom (Runtime -> Run all). On Kaggle turn Internet **On**. Outputs: `results/ngram.json`, `ckpt/ngram.pt`, and `ngram_10m.zip`.

In [ ]:
!pip -q install datasets tokenizers matplotlib

In [ ]:
NAME = "ngram"
STEPS = 4000   # lower (e.g. 2000) for a faster run
V, SEQ, EOS = 4096, 256, 0
import os, math, json, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# ---- data: TinyStories + BPE(4096) -> data/*.npy
def prep_data(out="data", n_train=300_000, n_val=5_000):
    if os.path.exists(f"{out}/train.npy"): return
    os.makedirs(out, exist_ok=True)
    from datasets import load_dataset
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    ds = load_dataset("roneneldan/TinyStories")
    tr = ds["train"].select(range(n_train))["text"]; va = ds["validation"].select(range(n_val))["text"]
    tok = Tokenizer(models.BPE()); tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    tok.train_from_iterator(tr[:100_000], trainers.BpeTrainer(
        vocab_size=V, special_tokens=["<eos>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    tok.save(f"{out}/tok.json")
    def enc(texts):
        parts = []
        for i in range(0, len(texts), 20_000):
            ids = []
            for e in tok.encode_batch(texts[i:i + 20_000]): ids.extend(e.ids); ids.append(EOS)
            parts.append(np.array(ids, dtype=np.uint16))
        return np.concatenate(parts)
    for n, t in [("train", tr), ("val", va)]:
        a = enc(t); np.save(f"{out}/{n}.npy", a); print(n, "tokens:", len(a))

def get_batch(data, bs, T, g=None):
    ix = torch.randint(0, len(data) - T - 1, (bs,), generator=g).to(data.device)
    idx = ix[:, None] + torch.arange(T + 1, device=data.device)
    b = data[idx].long(); return b[:, :-1], b[:, 1:]

prep_data()

In [ ]:
# ---- Kneser-Ney trigram (vectorised numpy)
NAMES = {'ngram': 'N-gram (KN 3-gram)'}
def run_ngram(tokens=20_000_000):
    os.makedirs("results", exist_ok=True)
    if os.path.exists("results/ngram.json"): return
    tr = np.load("data/train.npy")[:tokens].astype(np.int64); va = np.load("data/val.npy").astype(np.int64); V2 = V * V
    u3, c3 = np.unique(tr[:-2] * V2 + tr[1:-1] * V + tr[2:], return_counts=True)
    ab, bc = u3 // V, u3 % V2
    ab_keys, ab_i, ab_types = np.unique(ab, return_index=True, return_counts=True); ab_tot = np.add.reduceat(c3, ab_i)
    bc_keys, n_bc = np.unique(bc, return_counts=True)            # continuation counts N1+(. b c)
    b_keys, b_i, b_types = np.unique(bc_keys // V, return_index=True, return_counts=True); b_tot = np.add.reduceat(n_bc, b_i)
    uni = np.bincount(bc_keys % V, minlength=V).astype(np.float64); p_uni = (uni + 1) / (uni.sum() + V)
    disc = lambda c: (c == 1).sum() / ((c == 1).sum() + 2 * (c == 2).sum()); D3, D2 = disc(c3), disc(n_bc)
    def look(keys, q): i = np.minimum(np.searchsorted(keys, q), len(keys) - 1); return i, keys[i] == q
    a, b, c = va[:-2], va[1:-1], va[2:]; pu = p_uni[c]
    with np.errstate(all="ignore"):
        ib, fb = look(b_keys, b); ibc, fbc = look(bc_keys, b * V + c)
        p2 = np.where(fb, (np.where(fbc, np.maximum(n_bc[ibc] - D2, 0), 0.0) + D2 * b_types[ib] * pu) / b_tot[ib], pu)
        iab, fab = look(ab_keys, a * V + b); iabc, fabc = look(u3, a * V2 + b * V + c)
        p3 = np.where(fab, (np.where(fabc, np.maximum(c3[iabc] - D3, 0), 0.0) + D3 * ab_types[iab] * p2) / ab_tot[iab], p2)
    loss = float(-np.log(p3).mean())
    res = {"name": "ngram", "label": NAMES["ngram"], "params": int(len(u3) + len(bc_keys) + V), "val_loss": loss,
           "val_ppl": math.exp(loss), "val_bits_per_token": loss / math.log(2), "tok_per_s": None, "note": "params = stored n-gram entries"}
    json.dump(res, open("results/ngram.json", "w"), indent=1); print("[ngram]", res)

run_ngram()

In [ ]:
import json; r = json.load(open('results/ngram.json')); print(r)
# NOTE: 'params' = number of stored n-gram entries (not trainable parameters).

### Export for the portfolio

In [ ]:
import shutil, os; os.makedirs("portfolio", exist_ok=True); shutil.copy("results/ngram.json", "portfolio/"); shutil.make_archive("ngram_10m", "zip", "portfolio")